# NB4 — Circuit Decomposition

In NB3 we *found* the induction heads. Now we **explain and prove the circuit**:

1. **The mechanism** — how a **previous-token head** (L4H11) feeds the **induction heads** via
   *K-composition*.
2. **Ablation** — zero (and mean) out heads and watch the second-copy loss explode, proving the
   heads are *necessary*, not merely correlated.
3. **Direct logit attribution (DLA)** — measure how much each head pushes the *correct* next-token
   logit, so we can see which heads do the final writing.

This ties together everything: the residual stream (NB1), reading patterns (NB2), the induction
signature (NB3), and hooks that now *edit* activations rather than just read them.

## The mechanism: prev-token head → induction head (K-composition)

Induction needs two steps, done by two different heads in two different layers:

Consider the second copy, current token `X` at position `d`. `X` first appeared at position `j`, and
the token we want to predict is whatever sat at `j+1`.

- **Step 1 — previous-token head (early, L4H11).** At position `j+1`, this head attends one step back
  to `j` and copies information about *the token before me* (`X`) into the residual stream at `j+1`.
  So position `j+1` now carries a tag: "my predecessor was `X`."
- **Step 2 — induction head (later, L5/L6/L7).** At position `d`, its **query** encodes the current
  token `X`. Its **keys** read that tag the prev-token head wrote. The key at `j+1` says "predecessor
  = `X`", which matches the query → the induction head attends `d → j+1` and copies that token to the
  output as the prediction.

This is **K-composition**: the induction head's *keys* (the K in QK) are computed from a subspace of
the residual stream that an earlier head (the prev-token head) *wrote* to. The two heads compose
through the residual stream. **Prediction:** kill the prev-token head and the induction heads lose
the tag they key on — induction should break even though we didn't touch the induction heads
themselves. We'll test exactly that.

## 0. Setup and baseline

Rebuild the repeated-token setup from NB3 and define a helper that returns the **mean second-copy
loss** (our single number for "how well is induction working").

In [ ]:
import torch
from collections import defaultdict
from transformer_lens import HookedTransformer

torch.set_grad_enabled(False)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)
nL, nH = model.cfg.n_layers, model.cfg.n_heads

seq_len = 50
def gen_repeated_tokens(model, seq_len, batch=1):
    bos = torch.full((batch, 1), model.tokenizer.bos_token_id, dtype=torch.long)
    rand = torch.randint(0, model.cfg.d_vocab, (batch, seq_len), dtype=torch.long)
    return torch.cat([bos, rand, rand], dim=1).to(model.cfg.device)

rep_tokens = gen_repeated_tokens(model, seq_len, batch=20)

def second_copy_loss(fwd_hooks=[]):
    """Mean cross-entropy over the second-copy positions, optionally under ablation hooks."""
    loss = model.run_with_hooks(rep_tokens, return_type="loss", loss_per_token=True, fwd_hooks=fwd_hooks)
    return loss[:, seq_len:].mean().item()

baseline = second_copy_loss()
print(f"baseline second-copy loss: {baseline:.3f}")   # low -> induction is working

## 1. Ablation — proving necessity

**Ablation** = force an activation to some uninformative value and see how much the model breaks.
The bigger the damage, the more the model *relied* on that component.

We ablate a **head's output** by zeroing its slice of `hook_z` — the per-head value `z`
(`[batch, pos, head, d_head]`) *before* it's projected by `W_O`. Zeroing head `h`'s `z` removes that
head's entire contribution to the residual stream, cleanly switching the head off.

The hook signature is the `(activation, hook)` one you used in NB3 — but now the hook **returns a
modified tensor**, and TransformerLens splices it back into the forward pass. That return is the only
difference between a *reading* hook and an *editing* hook.

In [ ]:
def ablate_heads(heads):
    """Build fwd_hooks that ZERO the given (layer, head) outputs at hook_z."""
    by_layer = defaultdict(list)
    for L, H in heads:
        by_layer[L].append(H)

    def hook(z, hook):                       # z: [batch, pos, head, d_head]
        for h in by_layer[hook.layer()]:     # empty list -> no-op for untouched layers
            z[:, :, h, :] = 0.0
        return z                             # returning the edited tensor is what makes it stick

    return [(lambda name: name.endswith("hook_z"), hook)]

induction_heads = [(5, 5), (5, 1), (6, 9), (7, 10), (7, 2)]

print(f"baseline                : {baseline:.3f}")
print(f"ablate induction heads  : {second_copy_loss(ablate_heads(induction_heads)):.3f}   <- induction destroyed")

### Zero vs mean ablation

Zeroing is the bluntest choice. A subtler baseline is **mean ablation**: replace the head's output
with its *average* value (over batch and positions) instead of zero. The idea is to remove the
head's *input-dependent* signal while keeping its constant "bias" contribution, so you're testing the
information it carries rather than its mere presence. For induction heads the two give a similar
verdict — the loss still jumps — because their contribution is almost entirely input-dependent.

In [ ]:
def mean_ablate_heads(heads):
    """Replace each head's z with its mean over batch and position."""
    by_layer = defaultdict(list)
    for L, H in heads:
        by_layer[L].append(H)

    def hook(z, hook):                                   # z: [batch, pos, head, d_head]
        for h in by_layer[hook.layer()]:
            z[:, :, h, :] = z[:, :, h, :].mean(dim=(0, 1), keepdim=True)  # broadcast the mean back
        return z

    return [(lambda name: name.endswith("hook_z"), hook)]

print(f"zero-ablate induction heads: {second_copy_loss(ablate_heads(induction_heads)):.3f}")
print(f"mean-ablate induction heads: {second_copy_loss(mean_ablate_heads(induction_heads)):.3f}")

## 2. Practice — ablate the *upstream* head and watch the circuit break

This is the K-composition test. The prev-token head **L4H11** is *not* an induction head — it never
attends to "the token after the previous copy." But the mechanism says the induction heads depend on
the tag it writes. So ablating L4H11 alone should still damage induction.

**Your task**, using the `ablate_heads` helper already defined:
1. Zero-ablate **just L4H11** and print the second-copy loss.
2. As a control, zero-ablate one ordinary head that isn't part of the circuit — say **L3H0** — and
   print its loss.
3. Compare all three to `baseline`. You should see: **L4H11 clearly hurts** induction (K-composition
   confirmed), while the control barely moves the loss.

Hint: `ablate_heads([(4, 11)])` gives you the fwd_hooks; pass them to `second_copy_loss(...)`.

In [ ]:
# TODO(you):
# prev_token_loss = second_copy_loss(ablate_heads([(4, 11)]))
# control_loss    = second_copy_loss(ablate_heads([(3, 0)]))
# print baseline, prev_token_loss, control_loss and compare


<details>
<summary>Reference solution (open after you've tried)</summary>

```python
prev_token_loss = second_copy_loss(ablate_heads([(4, 11)]))
control_loss    = second_copy_loss(ablate_heads([(3, 0)]))

print(f"baseline            : {baseline:.3f}")
print(f"ablate L4H11 (prev) : {prev_token_loss:.3f}   <- upstream head; induction still suffers")
print(f"ablate L3H0 (control): {control_loss:.3f}   <- barely changes")
```

Expected: `baseline ~0.51`, `L4H11 ~0.76` (clearly worse), `L3H0 ~0.56` (near baseline). The
prev-token head matters *because* the induction heads key on what it writes — that's K-composition.

</details>

## 3. Direct logit attribution (DLA)

Ablation asks "what breaks if this head is gone?" DLA asks the complementary question: "how much does
each head **directly** push the *correct* next-token logit?"

The logit for a token is `(final residual stream) · W_U[:, token]`. Because the residual stream is a
**sum** over components (NB1!), each head's contribution to that logit is just *its* slice of the sum
dotted with the unembedding direction of the correct token. TransformerLens gives us the per-head
contributions directly:

- `cache.stack_head_results(layer=-1, pos_slice=...)` → each head's write into the final residual
  stream, shape `[n_heads_total, batch, pos, d_model]`.
- `cache.apply_ln_to_stack(...)` folds in the final LayerNorm scaling (so the dot product is exact).

We evaluate this on the **second-copy** prediction positions and dot with each position's correct
token.

In [ ]:
_, cache = model.run_with_cache(rep_tokens)

# Second-copy PREDICTION positions are seq_len .. 2*seq_len-1 (each predicts the next token).
pos_slice = slice(seq_len, -1)
per_head, labels = cache.stack_head_results(layer=-1, pos_slice=pos_slice, return_labels=True)
per_head = cache.apply_ln_to_stack(per_head, layer=-1, pos_slice=pos_slice)   # [comp, batch, pos, d_model]

correct = rep_tokens[:, seq_len + 1:]           # the token each of those positions should predict
correct_dirs = model.W_U[:, correct]            # [d_model, batch, pos]

# Each head's average contribution to the correct-token logit:
dla = torch.einsum("c b p d, d b p -> c b p", per_head, correct_dirs).mean(dim=(1, 2))

print("Top heads by direct logit attribution (second copy):")
for v, i in zip(*dla.topk(6)):
    print(f"  {labels[i]}: {v.item():+.3f}")

## The circuit, assembled

Putting the three tools together, here's the induction circuit in GPT-2 small, proven end to end:

| Evidence | Tool | Result |
|----------|------|--------|
| Induction heads exist | attention-pattern score (NB3) | L5H5, L5H1, L6H9, L7H10, L7H2 attend to "token after previous copy" |
| They're **necessary** | zero/mean ablation | ablating them: second-copy loss ~0.5 → ~5+ |
| The **prev-token head feeds them** | ablate upstream L4H11 | induction degrades without touching the induction heads → K-composition |
| They **write the answer** | direct logit attribution | L7H2 / L7H10 / L6H9 are among the top direct contributors to the correct logit |

Notice DLA and the attention score don't rank heads identically: the L5 heads score highest on
induction *attention* (they set the pattern up early), while the L6/L7 heads do more of the direct
*logit writing*. Both are part of the same circuit, playing different roles — exactly the kind of
nuance mechanistic interpretability exists to surface.

**You did it.** From "load a model" (NB0) to a proven multi-head circuit (NB4).

---
**Done?** Fill in the practice cell (ablate L4H11 vs the control) and ask me to review. That closes
out the course — after which the natural next steps are: try **path patching** to nail down the
K-composition subspace precisely, or repeat the whole analysis on a different behaviour (e.g. the IOI
circuit) to practice the workflow on your own.